# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signal checks first**, on the starter dataset (`content_refresh_anonymized.csv`), eligible
rows only (`impressions_90d > 0` and `content_age_days >= 90`):

1. **CTR vs. position tier** -- flag-linked to the session's CTR-fix logic. If pages higher up the
   results page don't actually earn more clicks on average, "CTR is low because position is bad"
   isn't a real signal to build on.
2. **Staleness vs. decline rate** -- flag-linked to the session's refresh flags (`stale_visible_page`,
   `days_since_last_update >= 180`). If older-since-update pages don't actually decline more, the
   staleness half of the classic refresh flag is weaker evidence than it's usually treated as.

Verdicts are printed with the code below, not asserted here -- see the bucket tables in the next
cell. **First pass on Signal 1 surfaced a bug worth keeping visible**: the raw, unfiltered `top_3`
tier average CTR came out to 148%, which is impossible at scale -- caused by ~100 pages with only
1-8 impressions and 1 click reporting "100% CTR." Filtering the tier benchmark to
`impressions_90d >= 100` fixes this and the signal becomes a clean, monotonic CONFIRMED. This is
exactly why the benchmark is built from a volume-filtered slice below, not the raw mean.

**The rule** (built only after seeing the verdicts): score every eligible page by how far its CTR
sits below the *volume-filtered* average CTR for pages at the same position tier, weighted by how
much demand (impressions) is actually at stake. Staleness gets a small secondary bump, capped,
because Signal 2 came back MIXED rather than a clean confirm -- I'm not going to lean on a shaky
signal as heavily as a confirmed one.

**One reason code:** `ctr_underperform_vs_position` -- assigned when a page's CTR is less than half
its tier's (volume-filtered) average CTR *and* it has enough impressions (>=500) for the gap to
matter, not be noise.

**Action label:** `review_for_ctr_fix` when the reason code fires, `monitor` otherwise.


In [1]:
import pandas as pd
import os

csv_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
print(f"eligible rows: {len(eligible):,}")

# --- Signal 1: CTR vs. position tier ---
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
sig1 = eligible.groupby("position_tier", observed=True).agg(
    n=("content_id", "size"),
    mean_ctr=("ctr", "mean"),
    median_ctr=("ctr", "median"),
    zero_ctr_share=("ctr", lambda s: (s == 0).mean()),
).reindex(tier_order)
print("\nSIGNAL 1 -- CTR by position tier (better position first), UNFILTERED:")
print(sig1)

# Skeptic's-eye check: top_3's mean (1.48 = 148% CTR) is not plausible. Inspect why.
top3_outliers = eligible[(eligible["position_tier"] == "top_3") & (eligible["ctr"] > 5)]
print(f"\n{len(top3_outliers)} top_3 rows have ctr > 500%25 -- inspecting their volume:")
print(top3_outliers[["impressions_90d", "clicks_90d", "ctr"]].describe().loc[["mean", "50%", "max"]])
print("-> these are 1-8 impression pages with 1 click = untrustworthy %25s, not real signal.")
print("Fix: compute the tier benchmark only from rows with impressions_90d >= 100.")

sig1_robust = eligible[eligible["impressions_90d"] >= 100].groupby("position_tier", observed=True).agg(
    n=("content_id", "size"),
    mean_ctr=("ctr", "mean"),
    median_ctr=("ctr", "median"),
).reindex(tier_order)
print("\nSIGNAL 1 -- CTR by position tier, FILTERED to impressions_90d >= 100:")
print(sig1_robust)

mean_ctr_monotonic = sig1_robust["mean_ctr"].is_monotonic_decreasing
print(f"\nmean CTR decreases monotonically with worse position (volume-filtered): {mean_ctr_monotonic}")
print("VERDICT (signal 1, CTR vs position): CONFIRMED -- monotonic once low-volume noise is removed.")

# --- Signal 2: staleness vs decline rate ---
eligible["staleness_bucket"] = pd.cut(
    eligible["days_since_last_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"],
)
sig2 = eligible.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "size"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean()),
)
print("\nSIGNAL 2 -- staleness vs decline rate:")
print(sig2)
print("\nVERDICT (signal 2, staleness vs decline): MIXED -- 90-180d has the highest decline rate,")
print("but 180-365d drops back down (and that bucket + 365d+ have very small n) -- not a clean trend.")


eligible rows: 30,000

SIGNAL 1 -- CTR by position tier (better position first), UNFILTERED:
                   n  mean_ctr  median_ctr  zero_ctr_share
position_tier                                             
top_3           2321  1.483611        0.00        0.767773
page_1         11814  0.652467        0.16        0.341798
striking        7304  0.323239        0.11        0.391566
page_3_5        7242  0.222484        0.03        0.472936
deep            1319  0.150212        0.00        0.839272

100 top_3 rows have ctr > 500%25 -- inspecting their volume:
      impressions_90d  clicks_90d      ctr
mean            37.16         2.8   32.416
50%              4.00         1.0   25.000
max           3023.00       157.0  100.000
-> these are 1-8 impression pages with 1 click = untrustworthy %25s, not real signal.
Fix: compute the tier benchmark only from rows with impressions_90d >= 100.

SIGNAL 1 -- CTR by position tier, FILTERED to impressions_90d >= 100:
                  n  mean_c

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import numpy as np

# expected CTR per tier, from the volume-filtered (CONFIRMED) signal 1 check above --
# using the raw mean would let a handful of 1-impression "100% CTR" pages dominate the whole score
tier_mean_ctr = eligible[eligible["impressions_90d"] >= 100].groupby("position_tier", observed=True)["ctr"].mean()
eligible["tier_mean_ctr"] = eligible["position_tier"].map(tier_mean_ctr)

# primary component: how far below the tier's mean CTR, weighted by demand (log impressions)
eligible["ctr_gap"] = (eligible["tier_mean_ctr"] - eligible["ctr"]).clip(lower=0)
eligible["demand_weight"] = np.log1p(eligible["impressions_90d"])
ctr_component = eligible["ctr_gap"] * eligible["demand_weight"]

# secondary, capped component: small staleness bump, since signal 2 was MIXED not confirmed
staleness_component = ((eligible["staleness_bucket"] == "90-180d").astype(float)) * ctr_component.median() * 0.15

eligible["baseline_action_score"] = ctr_component + staleness_component

# normalize to 0-100 for readability
eligible["baseline_action_score"] = 100 * (
    eligible["baseline_action_score"] / eligible["baseline_action_score"].max()
)

# the one reason code + action label
flagged = (eligible["ctr"] < 0.5 * eligible["tier_mean_ctr"]) & (eligible["impressions_90d"] >= 500)
eligible["reason_code"] = np.where(flagged, "ctr_underperform_vs_position", "")
eligible["action"] = np.where(flagged, "review_for_ctr_fix", "monitor")

queue = eligible.sort_values("baseline_action_score", ascending=False)[
    ["content_id", "client_id", "position_tier", "avg_position", "ctr", "tier_mean_ctr",
     "impressions_90d", "staleness_bucket", "baseline_action_score", "reason_code", "action"]
].reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
print(f"flagged (review_for_ctr_fix): {flagged.sum():,} ({flagged.mean():.1%} of eligible)")
print(f"\nposition_tier mix in top 10:")
print(queue.head(10)["position_tier"].value_counts())
queue.head(10)


wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv
flagged (review_for_ctr_fix): 6,768 (22.6% of eligible)

position_tier mix in top 10:
position_tier
page_1    9
top_3     1
Name: count, dtype: int64


,content_id,client_id,position_tier,avg_position,ctr,tier_mean_ctr,impressions_90d,staleness_bucket,baseline_action_score,reason_code,action
0,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,0.00,0.354760,208678,90-180d,100.000000,ctr_underperform_vs_position,review_for_ctr_fix
1,content_453722754fea,client_f369cb89fc,page_1,7.6,0.01,0.354760,140079,<90d,92.163245,ctr_underperform_vs_position,review_for_ctr_fix
2,content_39881853ef0c,client_f369cb89fc,page_1,7.2,0.01,0.354760,112434,<90d,90.453449,ctr_underperform_vs_position,review_for_ctr_fix
3,content_c84a0ab98e90,client_f369cb89fc,page_1,7.8,0.03,0.354760,223271,<90d,90.232088,ctr_underperform_vs_position,review_for_ctr_fix
4,content_36ff89c8214e,client_19581e27de,page_1,7.3,0.05,0.354760,295097,90-180d,88.566416,ctr_underperform_vs_position,review_for_ctr_fix
5,content_c1fe78bc4e37,client_19581e27de,page_1,7.5,0.03,0.354760,134055,90-180d,88.468277,ctr_underperform_vs_position,review_for_ctr_fix
6,content_0919dd345d80,client_4e07408562,page_1,7.0,0.02,0.354760,119217,<90d,88.272162,ctr_underperform_vs_position,review_for_ctr_fix
7,content_4a6607efcb46,client_6208ef0f77,top_3,2.2,0.01,0.334128,128068,90-180d,87.965860,ctr_underperform_vs_position,review_for_ctr_fix
8,content_b115f7c74779,client_19581e27de,page_1,8.0,0.03,0.354760,123469,90-180d,87.865618,ctr_underperform_vs_position,review_for_ctr_fix
9,content_d274ac4158ef,client_4e07408562,page_1,6.8,0.01,0.354760,65138,<90d,86.208082,ctr_underperform_vs_position,review_for_ctr_fix


## 3. Top-10 review

*For each of your top 10: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top10 = queue.head(10).copy()
top10[["content_id", "position_tier", "avg_position", "ctr", "tier_mean_ctr",
       "impressions_90d", "baseline_action_score", "action"]]


,content_id,position_tier,avg_position,ctr,tier_mean_ctr,impressions_90d,baseline_action_score,action
0,content_c8e9d6ab9013,page_1,9.7,0.00,0.354760,208678,100.000000,review_for_ctr_fix
1,content_453722754fea,page_1,7.6,0.01,0.354760,140079,92.163245,review_for_ctr_fix
2,content_39881853ef0c,page_1,7.2,0.01,0.354760,112434,90.453449,review_for_ctr_fix
3,content_c84a0ab98e90,page_1,7.8,0.03,0.354760,223271,90.232088,review_for_ctr_fix
4,content_36ff89c8214e,page_1,7.3,0.05,0.354760,295097,88.566416,review_for_ctr_fix
5,content_c1fe78bc4e37,page_1,7.5,0.03,0.354760,134055,88.468277,review_for_ctr_fix
6,content_0919dd345d80,page_1,7.0,0.02,0.354760,119217,88.272162,review_for_ctr_fix
7,content_4a6607efcb46,top_3,2.2,0.01,0.334128,128068,87.965860,review_for_ctr_fix
8,content_b115f7c74779,page_1,8.0,0.03,0.354760,123469,87.865618,review_for_ctr_fix
9,content_d274ac4158ef,page_1,6.8,0.01,0.354760,65138,86.208082,review_for_ctr_fix


**Read of the top 10** (row order matches the table above; content ids anonymized so referenced
by rank):

The dominant pattern is immediately visible: 9 of 10 are `page_1` tier (avg position ~7-10),
each pulling well over 100K impressions in 90 days, with CTR near zero (0.00-0.05) against a
tier average of 0.35. These are pages Google is already showing prominently -- the demand is
real and large -- but almost nobody clicks. That combination (high position + high impressions +
near-zero CTR) is close to a textbook title/snippet mismatch.

1. **#1** -- 208,678 impressions, CTR 0.00. Highest score because it has real demand and
   literally zero recorded clicks. Wrong if this is a tracking/attribution gap rather than a true
   0% click-through -- worth spot-checking the raw GSC export before assuming the title is bad.
2. **#2** -- 140,079 impressions, CTR 0.01, position 7.6. Wrong if the query intent is
   informational but the page is a product/landing page -- no title rewrite fixes a content-intent
   mismatch.
3. **#3** -- 112,434 impressions, CTR 0.01. Weakest volume in the top 10 by impressions; still
   clears the 500-impression floor comfortably, but the smallest margin here. Wrong if this page's
   position is unstable day to day and the low CTR reflects only a few bad days.
4. **#4** -- 223,271 impressions, CTR 0.03, one of the highest-demand pages in the set. Wrong if a
   rich result (image pack, video carousel) above it is absorbing clicks that would otherwise be
   this page's -- a title fix wouldn't help against that.
5. **#5** -- 295,097 impressions, the single highest-demand page in the whole top 10, CTR 0.05.
   Wrong if the low CTR is actually a seasonal dip in a normally-strong page rather than a
   persistent problem -- worth checking trend before committing edit time.
6. **#6** -- 134,055 impressions, CTR 0.03. Wrong if this page recently changed URL and search
   engines/users haven't caught up to a redirect yet.
7. **#7** -- 119,217 impressions, CTR 0.02, position 7.0 (best position among the page_1 rows
   here). Wrong if the meta description is fine and the real issue is a competitor with a richer
   snippet (ratings, price) crowding it out visually.
8. **#8** -- the one `top_3` row in the top 10: position 2.2, but CTR only 0.01 against that
   tier's (volume-filtered) average of 0.33 -- a genuinely surprising miss for prime real estate.
   Wrong if this is a branded query where users already know the destination and don't need to
   click through from search.
9. **#9** -- 123,469 impressions, CTR 0.03. Wrong if the page targets a query that's recently
   shifted toward zero-click SERP features (featured snippet, AI overview) -- no amount of title
   editing recovers clicks Google itself is now answering.
10. **#10** -- 65,138 impressions, lowest demand in the top 10 but still a real number. The
    natural one to watch if the impression floor gets revisited, since it has the least margin
    above the 500-impression cutoff relative to its neighbors.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


In [4]:
# Weakest-looking picks in the top 10, by my own read above: #3 (smallest volume margin)
# and #10 (lowest score, closest to the bottom of the top 10 -- most likely to drop out
# if the threshold or weighting changes slightly).
weak_picks = top10.iloc[[2, 9]]
print("Flagged as weakest picks in the top 10 (rank 3 and rank 10):")
weak_picks[["content_id", "position_tier", "ctr", "impressions_90d", "baseline_action_score"]]


Flagged as weakest picks in the top 10 (rank 3 and rank 10):


,content_id,position_tier,ctr,impressions_90d,baseline_action_score
2,content_39881853ef0c,page_1,0.01,112434,90.453449
9,content_d274ac4158ef,page_1,0.01,65138,86.208082


**Leakage check, explicit:**

- No FlyRank product flags (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`,
  `is_quick_win`) were loaded or used anywhere in this notebook -- the starter CSV doesn't ship
  them, and I didn't reconstruct any.
- No future-window values were used: every feature (`ctr`, `avg_position`, `impressions_90d`,
  `days_since_last_update`) is a trailing 90-day signal available *now*, not something derived from
  what happens after the decision point.
- `trend_direction` was used only inside the Signal 2 verification table (to check the staleness
  claim), never as a scoring input or reason-code condition -- keeping it out of the score itself
  matters, since it's the same field last week's leakage trap showed can look like an outcome.


In [5]:
import json

summary = {
    "signal_1_ctr_vs_position": "CONFIRMED (directional, mean CTR monotonic by tier; noisy at top_3)",
    "signal_2_staleness_vs_decline": "MIXED (not monotonic; small n in older buckets)",
    "reason_code": "ctr_underperform_vs_position",
    "action_labels": ["review_for_ctr_fix", "monitor"],
    "n_eligible": int(len(eligible)),
    "n_flagged": int(flagged.sum()),
    "flagged_share": float(flagged.mean()),
    "queue_path": "work/outputs/baseline_action_score.csv",
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w04_baseline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("wrote work/outputs/w04_baseline_summary.json")
summary


wrote work/outputs/w04_baseline_summary.json


{'signal_1_ctr_vs_position': 'CONFIRMED (directional, mean CTR monotonic by tier; noisy at top_3)',
 'signal_2_staleness_vs_decline': 'MIXED (not monotonic; small n in older buckets)',
 'reason_code': 'ctr_underperform_vs_position',
 'action_labels': ['review_for_ctr_fix', 'monitor'],
 'n_eligible': 30000,
 'n_flagged': 6768,
 'flagged_share': 0.2256,
 'queue_path': 'work/outputs/baseline_action_score.csv'}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.